Before running this file, `processed_data` should already be fully up to date.

This notebook is meant to mirror the workflow of `peak_analysis`, but specialized to **Cs-only Compton / backscatter analysis**.

It does the following:

1. loads only Cs runs (including the thickness-effect files),
2. background-subtracts the spectrum,
3. lets you define two one-peak fit windows interactively:
   - **backscatter peak**
   - **Compton edge**
4. fits each selected feature with a Gaussian + linear background,
5. extracts the fitted centers and uncertainties,
6. converts the fitted ADC positions and widths to **energy (keV)** using the Feb. 24 base calibration,
7. inflates the quoted energy uncertainties using the day-specific `daily_error_adjustments.csv`,
8. computes the **Compton-region length** and **average slope** between those two fitted centers in both ADC and keV,
9. saves both per-run and master CSV outputs.

Note: the Compton edge is not truly Gaussian, but this notebook treats it as a fitted feature proxy to find a consistent center.



In [1]:

# -----------------------------
# FORCE GUI BACKEND
# -----------------------------

import matplotlib

_backend_errors = []

for candidate in ["QtAgg", "TkAgg"]:
    try:
        matplotlib.use(candidate, force=True)
        print(f"Using backend: {candidate}")
        break
    except Exception as e:
        _backend_errors.append((candidate, repr(e)))
else:
    print("Could not start QtAgg/TkAgg. Falling back to default backend.")
    print("Tried:", _backend_errors)

Using backend: QtAgg


In [2]:

# -----------------------------
# IMPORTS
# -----------------------------

import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit

plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)

In [3]:
# -----------------------------
# FILE LOCATIONS
# -----------------------------

ROOT = Path("processed_data")
BACKGROUND_FILE = ROOT / "calibration" / "background_550PM_feb26_213PM_mar3.csv"

OUTPUT_ROOT = Path("compton_analysis_outputs")
PER_RUN_DIR = OUTPUT_ROOT / "per_run_csv"
PLOT_DIR = OUTPUT_ROOT / "plots"
MASTER_FILE = OUTPUT_ROOT / "compton_feature_fits_all.csv"

irrelevant_relpaths = {
    "calibration/background_550PM_feb26_213PM_mar3.csv",
}
irrelevant_set = {Path(p).as_posix() for p in irrelevant_relpaths}

def first_existing(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return Path(paths[0])

BASE_CAL_FILE = first_existing(
    "calibration_data/base_calibration_feb24.csv",
    "base_calibration_feb24.csv",
    "/mnt/data/base_calibration_feb24.csv",
)

DAILY_ERR_FILE = first_existing(
    "calibration_data/daily_error_adjustments.csv",
    "daily_error_adjustments.csv",
    "/mnt/data/daily_error_adjustments.csv",
)

def normalize_relpath(path, root=ROOT):
    path = Path(path)
    try:
        rel = path.relative_to(root)
    except ValueError:
        rel = path
    return rel.as_posix()

print("ROOT:", ROOT)
print("BACKGROUND_FILE:", BACKGROUND_FILE)
print("Background exists:", BACKGROUND_FILE.exists())
print("BASE_CAL_FILE:", BASE_CAL_FILE)
print("DAILY_ERR_FILE:", DAILY_ERR_FILE)
print("OUTPUT_ROOT:", OUTPUT_ROOT)



ROOT: processed_data
BACKGROUND_FILE: processed_data/calibration/background_550PM_feb26_213PM_mar3.csv
Background exists: True
BASE_CAL_FILE: calibration_data/base_calibration_feb24.csv
DAILY_ERR_FILE: calibration_data/daily_error_adjustments.csv
OUTPUT_ROOT: compton_analysis_outputs


In [4]:

# -----------------------------
# LOAD SPECTRUM / METADATA
# -----------------------------

def _extract_first_number(text):
    if text is None:
        return None
    match = re.search(r"[-+]?\d*\.?\d+", str(text))
    if match is None:
        return None
    try:
        return float(match.group())
    except ValueError:
        return None

def _month_token_to_day(token):
    token = str(token).lower().replace("_", "")
    mapping = {
        "feb24": "feb24",
        "feb26": "feb26",
        "mar3": "mar3",
        "mar5": "mar5",
    }
    return mapping.get(token)

def infer_day_from_path_or_metadata(relpath, metadata):
    rel = str(relpath).lower()
    for token in ["feb24", "feb26", "mar3", "mar5"]:
        if token in rel.replace("_", ""):
            return token

    start_time = str(metadata.get("Start Time", "")).lower()
    month_map = {
        "feb": "feb",
        "february": "feb",
        "mar": "mar",
        "march": "mar",
    }

    day_match = re.search(r"(feb(?:ruary)?|mar(?:ch)?)\s+(\d{1,2})", start_time)
    if day_match:
        month = month_map.get(day_match.group(1), "")
        day = int(day_match.group(2))
        token = f"{month}{day}"
        return _month_token_to_day(token)

    return None

def load_spectrum(csv_path):
    """
    Load a processed spectrum csv with preserved metadata and channel data.

    Expected processed structure:
    - metadata lines near the top
    - a line containing 'Channel Data:'
    - then a data table where:
        column 0 = channel / adc
        column 2 = counts
    """

    csv_path = Path(csv_path)

    with open(csv_path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    metadata = {}
    data_start_idx = None

    for i, line in enumerate(lines):
        stripped = line.strip()

        if stripped == "Channel Data:":
            data_start_idx = i + 1
            break

        if stripped.startswith("Elapsed Real Time:"):
            parts = stripped.split(",", 1)
            metadata["Elapsed Real Time"] = parts[1].strip() if len(parts) > 1 else ""
            continue

        if "Elapsed Real Time:" in stripped:
            value = stripped.split("Elapsed Real Time:", 1)[1]
            value = value.split(",", 1)[1].strip() if "," in value else value.strip()
            metadata["Elapsed Real Time"] = value
            continue

        if stripped.startswith("Start Time:"):
            metadata["Start Time"] = stripped.split("Start Time:", 1)[1].lstrip(",").strip()
            continue

        if stripped.startswith("End Time:"):
            metadata["End Time"] = stripped.split("End Time:", 1)[1].lstrip(",").strip()
            continue

        if "," in stripped:
            left, right = stripped.split(",", 1)
            left = left.strip().rstrip(":")
            right = right.strip()
            if left and right and left not in metadata:
                metadata[left] = right

    if data_start_idx is None:
        raise RuntimeError(f"Could not locate 'Channel Data:' block in {csv_path}")

    df = pd.read_csv(
        csv_path,
        skiprows=data_start_idx,
        header=None,
        names=["channel", "energy", "counts"],
    )

    df = df.dropna(subset=["channel", "counts"])
    x = pd.to_numeric(df["channel"], errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(df["counts"], errors="coerce").to_numpy(dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    ert = metadata.get("Elapsed Real Time", metadata.get("Elapsed Real Time (s)", None))
    ert_val = _extract_first_number(ert)
    metadata["Elapsed Real Time (s)"] = ert_val

    return x, y, metadata

def counts_to_rate(counts, live_time_s):
    counts = np.asarray(counts, dtype=float)
    if live_time_s is None or not np.isfinite(live_time_s) or live_time_s <= 0:
        raise RuntimeError("Invalid elapsed real time; cannot convert counts to count rate.")
    return counts / float(live_time_s)

def compute_background_subtracted_rate(sample_path, background_path=BACKGROUND_FILE):
    """
    Background subtraction happens BEFORE any feature fitting.
    """
    x_sample, counts_sample, meta_sample = load_spectrum(sample_path)
    x_bg, counts_bg, meta_bg = load_spectrum(background_path)

    if len(x_sample) != len(x_bg) or not np.allclose(x_sample, x_bg):
        raise RuntimeError(
            "Sample and background channel grids do not match. "
            "Background subtraction cannot be performed safely."
        )

    t_sample = meta_sample.get("Elapsed Real Time (s)")
    t_bg = meta_bg.get("Elapsed Real Time (s)")

    raw_rate = counts_to_rate(counts_sample, t_sample)
    background_rate = counts_to_rate(counts_bg, t_bg)
    adjusted_rate = raw_rate - background_rate

    return x_sample, raw_rate, background_rate, adjusted_rate, meta_sample, meta_bg

def normalize_preamp_value(value):
    text = str(value).lower().replace("_", "").replace(" ", "")
    if "100" in text:
        return 100
    if "0" in text:
        return 0
    try:
        return int(float(value))
    except Exception:
        return np.nan

def extract_preamp_from_path(relpath):
    lower = str(relpath).lower().replace("_", "")
    if "preamp100" in lower:
        return 100
    if "preamp0" in lower:
        return 0

    # All of the Cs geometry / ordinary runs in this data set appear to be preamp-0.
    if "cs" in lower:
        return 0

    return None

base_cal_df = pd.read_csv(BASE_CAL_FILE)
daily_err_df = pd.read_csv(DAILY_ERR_FILE)

base_cal_df["preamp_norm"] = base_cal_df["preamp"].map(normalize_preamp_value)
daily_err_df["preamp_norm"] = daily_err_df["preamp"].map(normalize_preamp_value)

BASE_CAL_MAP = {
    int(row["preamp_norm"]): row
    for _, row in base_cal_df.dropna(subset=["preamp_norm"]).iterrows()
}

DAILY_ERR_MAP = {
    (str(row["day"]).lower(), int(row["preamp_norm"])): row
    for _, row in daily_err_df.dropna(subset=["preamp_norm"]).iterrows()
}

print("Base calibration keys:", sorted(BASE_CAL_MAP))
print("Daily error keys:", sorted(DAILY_ERR_MAP))



Base calibration keys: [0, 100]
Daily error keys: [('feb26', 0), ('feb26', 100), ('mar3', 0), ('mar3', 100), ('mar5', 0), ('mar5', 100)]


In [5]:

# -----------------------------
# FIT MODELS
# -----------------------------

def gaussian(x, A, mu, sigma):
    sigma = abs(float(sigma))
    sigma = max(sigma, 1e-9)
    return A * np.exp(-((x - mu) ** 2) / (2 * sigma ** 2))

def gaussian_with_background(x, A, mu, sigma, m, b):
    return gaussian(x, A, mu, sigma) + m * x + b

def weighted_linear_fit(x, y, yerr=None):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if yerr is None:
        w = np.ones_like(y)
    else:
        yerr = np.asarray(yerr, dtype=float)
        yerr = np.where(np.isfinite(yerr) & (yerr > 0), yerr, 1.0)
        w = 1.0 / (yerr ** 2)

    coeffs, cov = np.polyfit(x, y, 1, w=np.sqrt(w), cov=True)
    slope, intercept = coeffs
    slope_err = np.sqrt(cov[0, 0]) if np.isfinite(cov[0, 0]) else np.nan
    intercept_err = np.sqrt(cov[1, 1]) if np.isfinite(cov[1, 1]) else np.nan

    yhat = slope * x + intercept
    resid = y - yhat

    if yerr is None:
        chi2 = np.sum(resid**2)
    else:
        chi2 = np.sum((resid / yerr) ** 2)

    dof = max(len(x) - 2, 1)
    redchi2 = chi2 / dof

    return {
        "slope": float(slope),
        "slope_err": float(slope_err),
        "intercept": float(intercept),
        "intercept_err": float(intercept_err),
        "chi2": float(chi2),
        "redchi2": float(redchi2),
        "yhat": yhat,
    }

def fit_single_feature(x, y, xmin, xmax, center_guess=None, label="feature", spectrum_label="current spectrum"):
    mask = (x >= float(xmin)) & (x <= float(xmax))
    xfit = np.asarray(x[mask], dtype=float)
    yfit = np.asarray(y[mask], dtype=float)

    if len(xfit) < 5:
        raise RuntimeError(f"{label}: fit window [{xmin}, {xmax}] is too small.")

    baseline = np.nanmedian([yfit[0], yfit[-1]])
    amp_guess = float(np.nanmax(yfit) - baseline)
    if center_guess is None:
        center_guess = float(xfit[np.nanargmax(yfit)])
    sigma_guess = max((float(xmax) - float(xmin)) / 8.0, 2.0)

    p0 = [amp_guess, float(center_guess), sigma_guess, 0.0, baseline]
    bounds = (
        [-np.inf, float(xmin), 1e-6, -np.inf, -np.inf],
        [ np.inf, float(xmax), np.inf,  np.inf,  np.inf],
    )

    yerr = np.sqrt(np.clip(yfit, 1e-6, None))

    popt, pcov = curve_fit(
        gaussian_with_background,
        xfit,
        yfit,
        p0=p0,
        sigma=yerr,
        absolute_sigma=True,
        maxfev=50000,
        bounds=bounds,
    )

    perr = np.sqrt(np.diag(pcov))

    A, mu, sigma, m, b = popt
    A_err, mu_err, sigma_err, m_err, b_err = perr

    area = abs(A) * abs(sigma) * np.sqrt(2.0 * np.pi)
    area_var = (np.sqrt(2.0 * np.pi) * sigma * A_err) ** 2 + (np.sqrt(2.0 * np.pi) * A * sigma_err) ** 2
    area_err = np.sqrt(max(area_var, 0.0))

    model = gaussian_with_background(xfit, *popt)
    resid = yfit - model
    chi2 = np.sum((resid / yerr) ** 2)
    dof = max(len(xfit) - len(popt), 1)
    redchi2 = chi2 / dof

    return {
        "label": label,
        "xmin": float(xmin),
        "xmax": float(xmax),
        "xfit": xfit,
        "yfit": yfit,
        "yerr": yerr,
        "popt": popt,
        "pcov": pcov,
        "model": model,
        "A": float(A),
        "A_err": float(A_err),
        "mu": float(mu),
        "mu_err": float(mu_err),
        "sigma": float(abs(sigma)),
        "sigma_err": float(abs(sigma_err)),
        "background_slope": float(m),
        "background_slope_err": float(m_err),
        "background_intercept": float(b),
        "background_intercept_err": float(b_err),
        "area": float(area),
        "area_err": float(area_err),
        "chi2": float(chi2),
        "redchi2": float(redchi2),
        "spectrum_label": spectrum_label,
    }

In [6]:
# -----------------------------
# ENERGY CALIBRATION HELPERS
# -----------------------------

def adc_to_energy(adc, adc_err, relpath):
    adc = np.asarray(adc, dtype=float)
    adc_err = np.asarray(adc_err, dtype=float)

    day = infer_day_from_path_or_metadata(relpath, {})
    preamp = extract_preamp_from_path(relpath)

    if preamp not in BASE_CAL_MAP:
        raise RuntimeError(f"No base calibration found for preamp={preamp} from file: {relpath}")

    row = BASE_CAL_MAP[preamp]
    slope = float(row["slope"])
    slope_err = float(row["slope_err"])
    offset = float(row["offset"])
    offset_err = float(row["offset_err"])

    energy = slope * adc + offset

    extra_sigma = 0.0
    if day is not None and (str(day).lower(), preamp) in DAILY_ERR_MAP:
        extra_sigma = float(DAILY_ERR_MAP[(str(day).lower(), preamp)]["extra_sigma_keV"])

    energy_err = np.sqrt(
        (slope * adc_err) ** 2 +
        (adc * slope_err) ** 2 +
        offset_err ** 2 +
        extra_sigma ** 2
    )

    return energy, energy_err, slope, slope_err, offset, offset_err, extra_sigma, day, preamp



In [7]:

# -----------------------------
# FILE LISTING
# -----------------------------

def is_cs_run(relpath):
    rel = Path(relpath).as_posix()
    lower = rel.lower()
    name = Path(rel).name.lower()

    if rel in irrelevant_set:
        return False

    # thickness effects are all Cs runs in your workflow
    if "thickness_effects" in lower:
        return True

    # standard source runs
    if "/cs137/" in lower:
        return True

    # calibration Cs files
    if name.startswith("cs_"):
        return True

    return False

def get_candidate_files(root=ROOT):
    csv_files = sorted(root.rglob("*.csv"))
    kept = []

    for file in csv_files:
        rel = normalize_relpath(file, root)

        if ".ipynb_checkpoints" in rel:
            continue

        if is_cs_run(rel):
            kept.append(file)

    return kept

candidate_files = get_candidate_files()

print("Number of Cs candidate files:", len(candidate_files))
for i, file in enumerate(candidate_files):
    print(f"[{i}] {normalize_relpath(file)}")

Number of Cs candidate files: 19
[0] calibration/daily/Cs_21.9C_preamp0_mar5.csv
[1] calibration/daily/Cs_22.0C_preamp0_mar3.csv
[2] calibration/daily/Cs_22.4C_preamp0_feb26.csv
[3] calibration/feb_24/preamp_0/Cs_22.1C_preamp0_feb24.csv
[4] cs gain setting/Cs_possible_calcurve_feb_24.csv
[5] cs gain setting/Cs_trial_feb_19.csv
[6] data/feb_26/cs137/Cs_22.4C_feb26_run1.csv
[7] data/mar_3/cs137/Cs_22.5C_mar3_run1.csv
[8] data/mar_5/cs137/Cs_21.9C_mar5_run1.csv
[9] data/thickness_effects_behind/Al thickness/Al_16cmpmt_1.3cmthick.csv
[10] data/thickness_effects_behind/Al thickness/Al_16cmpmt_14.7cmthick.csv
[11] data/thickness_effects_behind/Al thickness/Al_16cmpmt_2.3cmthick.csv
[12] data/thickness_effects_behind/Pb thickness/Pb_13.2cmpmt_0.1cmthick.csv
[13] data/thickness_effects_behind/Pb thickness/Pb_13.2cmpmt_14.5cmthick.csv
[14] data/thickness_effects_behind/Pb thickness/Pb_13.2cmpmt_7.5cmthick.csv
[15] data/thickness_effects_behind/Pb thickness/Pb_15.3cmpmt_4.8cmthick.csv
[16] data/

In [8]:

# -----------------------------
# INTERACTIVE FEATURE SELECTOR
# -----------------------------

FEATURE_STATE = {
    "x": None,
    "y": None,
    "title": None,
    "fig": None,
    "ax": None,
    "cid_click": None,
    "cid_key": None,
    "temp_artists": [],
    "status_text": None,
    "feature_specs": [],
    "pending_feature_order": ["backscatter_peak", "compton_edge"],
    "pending_clicks": [],
}

def nearest_index(xarr, xclick):
    return int(np.argmin(np.abs(np.asarray(xarr, dtype=float) - float(xclick))))

def _clear_feature_artists():
    for artist in FEATURE_STATE["temp_artists"]:
        try:
            artist.remove()
        except Exception:
            pass
    FEATURE_STATE["temp_artists"] = []

def _status_message():
    specs = FEATURE_STATE["feature_specs"]
    if len(specs) >= 2:
        return "Stored both features. Press q to close or r to reset."
    current = FEATURE_STATE["pending_feature_order"][len(specs)]
    step = len(FEATURE_STATE["pending_clicks"]) + 1
    prompts = {
        1: "left boundary",
        2: "center guess",
        3: "right boundary",
    }
    return (
        f"Selecting {current}: click {prompts[step]}. "
        "Keys: r=reset current/all, q=close"
    )

def _draw_feature_preview():
    _clear_feature_artists()

    ax = FEATURE_STATE["ax"]
    x = FEATURE_STATE["x"]
    pending = FEATURE_STATE["pending_clicks"]

    for xc in pending:
        idx = nearest_index(x, xc)
        artist = ax.axvline(x[idx], linestyle="--", alpha=0.7)
        FEATURE_STATE["temp_artists"].append(artist)

    for spec in FEATURE_STATE["feature_specs"]:
        for key, style in [("xmin", ":"), ("center_guess", "--"), ("xmax", ":")]:
            artist = ax.axvline(spec[key], linestyle=style, alpha=0.9)
            FEATURE_STATE["temp_artists"].append(artist)
        txt = ax.text(
            spec["center_guess"],
            ax.get_ylim()[1] * 0.85,
            spec["feature_label"],
            rotation=90,
            va="top",
            ha="center",
        )
        FEATURE_STATE["temp_artists"].append(txt)

    FEATURE_STATE["status_text"].set_text(_status_message())
    FEATURE_STATE["fig"].canvas.draw_idle()

def _store_pending_feature():
    if len(FEATURE_STATE["pending_clicks"]) != 3:
        return

    xs = sorted(float(v) for v in FEATURE_STATE["pending_clicks"])
    xmin, xmax = xs[0], xs[2]
    center_guess = float(FEATURE_STATE["pending_clicks"][1])

    if center_guess < xmin or center_guess > xmax:
        center_guess = 0.5 * (xmin + xmax)

    feature_label = FEATURE_STATE["pending_feature_order"][len(FEATURE_STATE["feature_specs"])]

    FEATURE_STATE["feature_specs"].append({
        "feature_label": feature_label,
        "xmin": xmin,
        "center_guess": center_guess,
        "xmax": xmax,
    })
    FEATURE_STATE["pending_clicks"] = []

def on_feature_click(event):
    if event.inaxes != FEATURE_STATE["ax"]:
        return
    if event.xdata is None:
        return
    if len(FEATURE_STATE["feature_specs"]) >= 2:
        return

    FEATURE_STATE["pending_clicks"].append(float(event.xdata))

    if len(FEATURE_STATE["pending_clicks"]) == 3:
        _store_pending_feature()

    _draw_feature_preview()

def on_feature_key(event):
    if event.key == "r":
        if FEATURE_STATE["pending_clicks"]:
            FEATURE_STATE["pending_clicks"] = []
        elif FEATURE_STATE["feature_specs"]:
            FEATURE_STATE["feature_specs"].pop()
        _draw_feature_preview()

    elif event.key == "q":
        plt.close(FEATURE_STATE["fig"])

def launch_feature_selector(x, y, title):
    FEATURE_STATE["x"] = np.asarray(x, dtype=float)
    FEATURE_STATE["y"] = np.asarray(y, dtype=float)
    FEATURE_STATE["title"] = title
    FEATURE_STATE["feature_specs"] = []
    FEATURE_STATE["pending_clicks"] = []

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(x, y)
    ax.set_xlabel("ADC Channel")
    ax.set_ylabel("Background-subtracted rate (counts/s)")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

    txt = ax.text(
        0.01, 0.99, "", transform=ax.transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
    )

    FEATURE_STATE["fig"] = fig
    FEATURE_STATE["ax"] = ax
    FEATURE_STATE["status_text"] = txt
    FEATURE_STATE["cid_click"] = fig.canvas.mpl_connect("button_press_event", on_feature_click)
    FEATURE_STATE["cid_key"] = fig.canvas.mpl_connect("key_press_event", on_feature_key)

    _draw_feature_preview()
    plt.show()

def get_feature_specs():
    return list(FEATURE_STATE["feature_specs"])

def clear_feature_state():
    fig = FEATURE_STATE.get("fig")
    if fig is not None:
        try:
            fig.canvas.mpl_disconnect(FEATURE_STATE["cid_click"])
        except Exception:
            pass
        try:
            fig.canvas.mpl_disconnect(FEATURE_STATE["cid_key"])
        except Exception:
            pass

    FEATURE_STATE.update({
        "x": None,
        "y": None,
        "title": None,
        "fig": None,
        "ax": None,
        "cid_click": None,
        "cid_key": None,
        "temp_artists": [],
        "status_text": None,
        "feature_specs": [],
        "pending_clicks": [],
    })

In [261]:
# -----------------------------
# START: CHOOSE CURRENT FILE
# -----------------------------

file_index = int(input("Enter file index from the list above: "))

current_file = candidate_files[file_index]
current_rel = normalize_relpath(current_file)

(
    x_current,
    raw_rate_current,
    background_rate_current,
    adjusted_rate_current,
    sample_meta_current,
    background_meta_current,
) = compute_background_subtracted_rate(current_file, background_path=BACKGROUND_FILE)

current_day = infer_day_from_path_or_metadata(current_rel, sample_meta_current)
current_preamp = extract_preamp_from_path(current_rel)

xerr_zero_current = np.zeros_like(x_current, dtype=float)
x_energy_current, x_energy_current_err, cal_slope_current, cal_slope_err_current, cal_offset_current, cal_offset_err_current, cal_extra_sigma_current, _, _ = adc_to_energy(
    x_current,
    xerr_zero_current,
    current_rel,
)

print("Current file:", current_rel)
print("Inferred day:", current_day)
print("Inferred preamp:", current_preamp)
print("Sample elapsed real time (s):", sample_meta_current.get("Elapsed Real Time (s)"))
print(f"Calibration: E = ({cal_slope_current:.6f} ± {cal_slope_err_current:.6f}) * ADC + ({cal_offset_current:.6f} ± {cal_offset_err_current:.6f}) keV")
print(f"Extra daily sigma (keV): {cal_extra_sigma_current:.6f}")



Enter file index from the list above:  18


Current file: data/thickness_effects_in_front/Pb_11cmpmt_tosource_5cmthick.csv
Inferred day: mar3
Inferred preamp: 0
Sample elapsed real time (s): 912.0
Calibration: E = (1.317572 ± 0.004624) * ADC + (-3.916731 ± 2.043277) keV
Extra daily sigma (keV): 0.000000


In [262]:
# -----------------------------
# DOCUMENTATION PLOT
# -----------------------------

fig_doc, ax_doc = plt.subplots(figsize=(10, 6))

ax_doc.plot(x_energy_current, raw_rate_current, label="Raw sample rate")
ax_doc.plot(x_energy_current, background_rate_current, label="Background rate")
ax_doc.plot(x_energy_current, adjusted_rate_current, label="Background-subtracted rate")

ax_doc.set_xlabel("Energy (keV)")
ax_doc.set_ylabel("Rate (counts/s)")
ax_doc.set_title(f"{current_rel}\nRaw vs background vs background-subtracted")
ax_doc.grid(True, alpha=0.3)
ax_doc.legend()

plt.show()



In [263]:

# -----------------------------
# RUN FEATURE SELECTOR
# -----------------------------

launch_feature_selector(
    x_current,
    adjusted_rate_current,
    title=f"{current_rel} | click backscatter then Compton edge"
)

In [264]:

# -----------------------------
# INSPECT FEATURE SPECS
# -----------------------------

feature_specs_current = get_feature_specs()

print("feature_specs_current:")
for spec in feature_specs_current:
    print(spec)

print("Number of stored features:", len(feature_specs_current))

feature_specs_current:
{'feature_label': 'backscatter_peak', 'xmin': 30.03000000000003, 'center_guess': 70.95000000000005, 'xmax': 85.47000000000003}
{'feature_label': 'compton_edge', 'xmin': 127.71000000000004, 'center_guess': 152.79000000000002, 'xmax': 183.15000000000003}
Number of stored features: 2


In [265]:

# -----------------------------
# OPTIONAL TYPED FALLBACK
# -----------------------------

# Only run this if you want to overwrite feature_specs_current manually.

def manual_feature_entry():
    specs = []
    for label in ["backscatter_peak", "compton_edge"]:
        print(f"\n{label}")
        xmin = float(input("xmin: "))
        center_guess = float(input("center guess: "))
        xmax = float(input("xmax: "))

        if xmax < xmin:
            xmin, xmax = xmax, xmin

        if center_guess < xmin or center_guess > xmax:
            center_guess = 0.5 * (xmin + xmax)

        specs.append({
            "feature_label": label,
            "xmin": xmin,
            "center_guess": center_guess,
            "xmax": xmax,
        })
    return specs

# feature_specs_current = manual_feature_entry()
# feature_specs_current

In [266]:

# -----------------------------
# SANITY CHECK
# -----------------------------

if "feature_specs_current" not in globals():
    raise RuntimeError("feature_specs_current does not exist. Run the selector cell first.")

if len(feature_specs_current) != 2:
    raise RuntimeError(
        "You should store exactly two features: "
        "backscatter_peak and compton_edge."
    )

labels = {spec["feature_label"] for spec in feature_specs_current}
required = {"backscatter_peak", "compton_edge"}
if labels != required:
    raise RuntimeError(f"Expected labels {required}, but got {labels}")

print("Ready to fit.")

Ready to fit.


In [267]:
# -----------------------------
# FIT BOTH FEATURES
# -----------------------------

fit_results_by_label = {}

for spec in feature_specs_current:
    label = spec["feature_label"]
    fit_results_by_label[label] = fit_single_feature(
        x_current,
        adjusted_rate_current,
        xmin=spec["xmin"],
        xmax=spec["xmax"],
        center_guess=spec["center_guess"],
        label=label,
        spectrum_label=current_rel,
    )

for label, res in fit_results_by_label.items():
    center_keV, center_keV_err, _, _, _, _, _, _, _ = adc_to_energy(
        res["mu"], res["mu_err"], current_rel
    )
    sigma_keV = abs(cal_slope_current) * res["sigma"]
    sigma_keV_err = np.sqrt(
        (res["sigma"] * cal_slope_err_current) ** 2 +
        (cal_slope_current * res["sigma_err"]) ** 2
    )

    res["center_keV"] = float(np.asarray(center_keV))
    res["center_keV_err"] = float(np.asarray(center_keV_err))
    res["sigma_keV"] = float(np.asarray(sigma_keV))
    res["sigma_keV_err"] = float(np.asarray(sigma_keV_err))

for label, res in fit_results_by_label.items():
    print("\n", label)
    print(f"  center = {res['mu']:.3f} ± {res['mu_err']:.3f} ADC")
    print(f"         = {res['center_keV']:.3f} ± {res['center_keV_err']:.3f} keV")
    print(f"  sigma  = {res['sigma']:.3f} ± {res['sigma_err']:.3f} ADC")
    print(f"         = {res['sigma_keV']:.3f} ± {res['sigma_keV_err']:.3f} keV")
    print(f"  area   = {res['area']:.3f} ± {res['area_err']:.3f}")
    print(f"  redchi2 = {res['redchi2']:.3f}")




 backscatter_peak
  center = 67.379 ± 3.861 ADC
         = 84.860 ± 5.491 keV
  sigma  = 4.779 ± 4.432 ADC
         = 6.297 ± 5.839 keV
  area   = 7.397 ± 8.946
  redchi2 = 0.005

 compton_edge
  center = 154.486 ± 3.931 ADC
         = 199.630 ± 5.614 keV
  sigma  = 10.316 ± 5.197 ADC
         = 13.592 ± 6.847 keV
  area   = 36.552 ± 22.316
  redchi2 = 0.015


In [268]:
# -----------------------------
# COMPUTE COMPTON-REGION METRICS
# -----------------------------

back = fit_results_by_label["backscatter_peak"]
edge = fit_results_by_label["compton_edge"]

mu_back = float(back["mu"])
mu_back_err = float(back["mu_err"])
mu_edge = float(edge["mu"])
mu_edge_err = float(edge["mu_err"])

if mu_edge <= mu_back:
    raise RuntimeError(
        "The fitted Compton-edge center is not to the right of the backscatter center. "
        "Re-check the selected windows."
    )

region_mask = (x_current >= mu_back) & (x_current <= mu_edge)
x_region = np.asarray(x_current[region_mask], dtype=float)
y_region = np.asarray(adjusted_rate_current[region_mask], dtype=float)
y_region_err = np.sqrt(np.clip(y_region, 1e-6, None))

if len(x_region) < 3:
    raise RuntimeError("Too few points in the fitted Compton region.")

line_fit = weighted_linear_fit(x_region, y_region, yerr=y_region_err)

compton_length_adc = mu_edge - mu_back
compton_length_adc_err = np.sqrt(mu_back_err**2 + mu_edge_err**2)

back_keV = back["center_keV"]
back_keV_err = back["center_keV_err"]
edge_keV = edge["center_keV"]
edge_keV_err = edge["center_keV_err"]

compton_length_keV = edge_keV - back_keV
compton_length_keV_err = np.sqrt(back_keV_err**2 + edge_keV_err**2)

delta_y = (line_fit["slope"] * mu_edge + line_fit["intercept"]) - (line_fit["slope"] * mu_back + line_fit["intercept"])
average_drop_per_adc = delta_y / compton_length_adc

compton_slope_per_keV = line_fit["slope"] / cal_slope_current
compton_slope_per_keV_err = np.sqrt(
    (line_fit["slope_err"] / cal_slope_current) ** 2 +
    ((line_fit["slope"] * cal_slope_err_current) / (cal_slope_current ** 2)) ** 2
) if cal_slope_current != 0 else np.nan

average_drop_per_keV = average_drop_per_adc / cal_slope_current
average_drop_per_keV_err = np.sqrt(
    (line_fit["slope_err"] / cal_slope_current) ** 2 +
    ((average_drop_per_adc * cal_slope_err_current) / (cal_slope_current ** 2)) ** 2
) if cal_slope_current != 0 else np.nan

print(f"Backscatter center = {mu_back:.3f} ± {mu_back_err:.3f} ADC = {back_keV:.3f} ± {back_keV_err:.3f} keV")
print(f"Compton-edge center = {mu_edge:.3f} ± {mu_edge_err:.3f} ADC = {edge_keV:.3f} ± {edge_keV_err:.3f} keV")
print(f"Compton-region length = {compton_length_adc:.3f} ± {compton_length_adc_err:.3f} ADC")
print(f"                      = {compton_length_keV:.3f} ± {compton_length_keV_err:.3f} keV")
print(f"Region slope = {line_fit['slope']:.6f} ± {line_fit['slope_err']:.6f} (counts/s)/ADC")
print(f"             = {compton_slope_per_keV:.6f} ± {compton_slope_per_keV_err:.6f} (counts/s)/keV")
print(f"Average drop per ADC across region = {average_drop_per_adc:.6f} (counts/s)/ADC")
print(f"Average drop per keV across region = {average_drop_per_keV:.6f} ± {average_drop_per_keV_err:.6f} (counts/s)/keV")
print(f"Compton-region line-fit redchi2 = {line_fit['redchi2']:.3f}")



Backscatter center = 67.379 ± 3.861 ADC = 84.860 ± 5.491 keV
Compton-edge center = 154.486 ± 3.931 ADC = 199.630 ± 5.614 keV
Compton-region length = 87.107 ± 5.510 ADC
                      = 114.770 ± 7.853 keV
Region slope = 0.008833 ± 0.001158 (counts/s)/ADC
             = 0.006704 ± 0.000879 (counts/s)/keV
Average drop per ADC across region = 0.008833 (counts/s)/ADC
Average drop per keV across region = 0.006704 ± 0.000879 (counts/s)/keV
Compton-region line-fit redchi2 = 0.063


In [269]:
# -----------------------------
# PLOT FITTED RESULT
# -----------------------------

fig_fit, ax_fit = plt.subplots(figsize=(11, 6))

ax_fit.plot(x_energy_current, adjusted_rate_current, label="Background-subtracted spectrum")

for label, res in fit_results_by_label.items():
    xfit_keV, _, _, _, _, _, _, _, _ = adc_to_energy(
        res["xfit"], np.zeros_like(res["xfit"], dtype=float), current_rel
    )
    ax_fit.plot(xfit_keV, res["model"], linewidth=2.2, label=f"{label} fit")
    ax_fit.axvline(res["center_keV"], linestyle="--", alpha=0.9)
    ax_fit.errorbar(
        [res["center_keV"]],
        [res["A"]],
        xerr=[res["center_keV_err"]],
        yerr=[res["A_err"]],
        fmt="o",
        capsize=3,
        label=f"{label} center ± error",
    )
    ax_fit.text(res["center_keV"], np.max(res["model"]), label, rotation=90, va="bottom", ha="center")

x_region_keV, _, _, _, _, _, _, _, _ = adc_to_energy(
    x_region, np.zeros_like(x_region, dtype=float), current_rel
)
ax_fit.plot(x_region_keV, line_fit["yhat"], label="Compton-region line fit")
ax_fit.axvspan(back_keV, edge_keV, alpha=0.15)

ax_fit.set_xlabel("Energy (keV)")
ax_fit.set_ylabel("Background-subtracted rate (counts/s)")
ax_fit.set_title(f"{current_rel}\nBackscatter + Compton-edge analysis")
ax_fit.grid(True, alpha=0.3)
ax_fit.legend()

plt.show()



In [270]:
# -----------------------------
# BUILD RESULTS DATAFRAME
# -----------------------------

rows = []

for label, res in fit_results_by_label.items():
    rows.append({
        "file": current_rel,
        "day": current_day,
        "preamp": current_preamp,
        "feature_label": label,
        "center_ADC": res["mu"],
        "center_ADC_err": res["mu_err"],
        "center_keV": res["center_keV"],
        "center_keV_err": res["center_keV_err"],
        "sigma_ADC": res["sigma"],
        "sigma_ADC_err": res["sigma_err"],
        "sigma_keV": res["sigma_keV"],
        "sigma_keV_err": res["sigma_keV_err"],
        "amplitude_counts_per_s": res["A"],
        "amplitude_counts_per_s_err": res["A_err"],
        "gaussian_area_counts": res["area"],
        "gaussian_area_counts_err": res["area_err"],
        "fit_xmin": res["xmin"],
        "fit_xmax": res["xmax"],
        "background_slope": res["background_slope"],
        "background_slope_err": res["background_slope_err"],
        "background_intercept": res["background_intercept"],
        "background_intercept_err": res["background_intercept_err"],
        "fit_chi2": res["chi2"],
        "fit_redchi2": res["redchi2"],
        "calibration_slope_keV_per_ADC": cal_slope_current,
        "calibration_slope_err": cal_slope_err_current,
        "calibration_offset_keV": cal_offset_current,
        "calibration_offset_err": cal_offset_err_current,
        "calibration_extra_sigma_keV": cal_extra_sigma_current,
        "compton_region_length_ADC": compton_length_adc,
        "compton_region_length_ADC_err": compton_length_adc_err,
        "compton_region_length_keV": compton_length_keV,
        "compton_region_length_keV_err": compton_length_keV_err,
        "compton_region_slope_counts_per_s_per_ADC": line_fit["slope"],
        "compton_region_slope_err_per_ADC": line_fit["slope_err"],
        "compton_region_slope_counts_per_s_per_keV": compton_slope_per_keV,
        "compton_region_slope_err_per_keV": compton_slope_per_keV_err,
        "compton_region_intercept": line_fit["intercept"],
        "compton_region_intercept_err": line_fit["intercept_err"],
        "compton_region_redchi2": line_fit["redchi2"],
        "compton_region_average_drop_per_ADC": average_drop_per_adc,
        "compton_region_average_drop_per_keV": average_drop_per_keV,
        "compton_region_average_drop_per_keV_err": average_drop_per_keV_err,
    })

results_df = pd.DataFrame(rows)

preferred_cols = [
    "file",
    "day",
    "preamp",
    "feature_label",
    "center_ADC",
    "center_ADC_err",
    "center_keV",
    "center_keV_err",
    "sigma_ADC",
    "sigma_ADC_err",
    "sigma_keV",
    "sigma_keV_err",
    "amplitude_counts_per_s",
    "amplitude_counts_per_s_err",
    "gaussian_area_counts",
    "gaussian_area_counts_err",
    "fit_xmin",
    "fit_xmax",
    "background_slope",
    "background_slope_err",
    "background_intercept",
    "background_intercept_err",
    "fit_chi2",
    "fit_redchi2",
    "calibration_slope_keV_per_ADC",
    "calibration_slope_err",
    "calibration_offset_keV",
    "calibration_offset_err",
    "calibration_extra_sigma_keV",
    "compton_region_length_ADC",
    "compton_region_length_ADC_err",
    "compton_region_length_keV",
    "compton_region_length_keV_err",
    "compton_region_slope_counts_per_s_per_ADC",
    "compton_region_slope_err_per_ADC",
    "compton_region_slope_counts_per_s_per_keV",
    "compton_region_slope_err_per_keV",
    "compton_region_intercept",
    "compton_region_intercept_err",
    "compton_region_redchi2",
    "compton_region_average_drop_per_ADC",
    "compton_region_average_drop_per_keV",
    "compton_region_average_drop_per_keV_err",
]

results_df = results_df[[c for c in preferred_cols if c in results_df.columns]]
display(results_df)



,file,day,preamp,feature_label,center_ADC,center_ADC_err,center_keV,center_keV_err,sigma_ADC,sigma_ADC_err,sigma_keV,sigma_keV_err,amplitude_counts_per_s,amplitude_counts_per_s_err,gaussian_area_counts,gaussian_area_counts_err,fit_xmin,fit_xmax,background_slope,background_slope_err,background_intercept,background_intercept_err,fit_chi2,fit_redchi2,calibration_slope_keV_per_ADC,calibration_slope_err,calibration_offset_keV,calibration_offset_err,calibration_extra_sigma_keV,compton_region_length_ADC,compton_region_length_ADC_err,compton_region_length_keV,compton_region_length_keV_err,compton_region_slope_counts_per_s_per_ADC,compton_region_slope_err_per_ADC,compton_region_slope_counts_per_s_per_keV,compton_region_slope_err_per_keV,compton_region_intercept,compton_region_intercept_err,compton_region_redchi2,compton_region_average_drop_per_ADC,compton_region_average_drop_per_keV,compton_region_average_drop_per_keV_err
0,data/thickness_effects_in_front/Pb_11cmpmt_tos...,mar3,0,backscatter_peak,67.379317,3.861092,84.860374,5.491115,4.779118,4.431827,6.296833,5.839294,0.617437,0.479455,7.396564,8.946280,30.03,85.47,0.005411,0.008662,0.303660,0.432197,0.271019,0.005420,1.317572,0.004624,-3.916731,2.043277,0.0,87.106904,5.510347,114.769621,7.85296,0.008833,0.001158,0.006704,0.000879,0.038104,0.125852,0.062872,0.008833,0.006704,0.000879
1,data/thickness_effects_in_front/Pb_11cmpmt_tos...,mar3,0,compton_edge,154.486221,3.931398,199.629995,5.613968,10.316312,5.196814,13.592484,6.847343,1.413488,0.487555,36.551621,22.315595,127.71,183.15,-0.006157,0.011142,1.690612,1.914348,0.748876,0.014684,1.317572,0.004624,-3.916731,2.043277,0.0,87.106904,5.510347,114.769621,7.85296,0.008833,0.001158,0.006704,0.000879,0.038104,0.125852,0.062872,0.008833,0.006704,0.000879


In [271]:

# -----------------------------
# SAVE PER-RUN RESULTS
# -----------------------------

PER_RUN_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

out_file = PER_RUN_DIR / f"{Path(current_rel).stem}_compton_features.csv"
results_df.to_csv(out_file, index=False)

fit_plot_file = PLOT_DIR / f"{Path(current_rel).stem}_compton_features.png"
fig_fit.savefig(fit_plot_file, dpi=200, bbox_inches="tight")

print("Saved per-run results to:", out_file)
print("Saved plot to:", fit_plot_file)

Saved per-run results to: compton_analysis_outputs/per_run_csv/Pb_11cmpmt_tosource_5cmthick_compton_features.csv
Saved plot to: compton_analysis_outputs/plots/Pb_11cmpmt_tosource_5cmthick_compton_features.png


In [272]:

# -----------------------------
# APPEND TO MASTER FILE
# -----------------------------

if MASTER_FILE.exists():
    old_df = pd.read_csv(MASTER_FILE)
    old_df = old_df[old_df["file"] != current_rel]
    combined_df = pd.concat([old_df, results_df], ignore_index=True)
else:
    combined_df = results_df.copy()

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
combined_df.to_csv(MASTER_FILE, index=False)

print("Updated master file:", MASTER_FILE)
print("Total rows in master file:", len(combined_df))
display(combined_df.tail())

Updated master file: compton_analysis_outputs/compton_feature_fits_all.csv
Total rows in master file: 38


,file,day,preamp,feature_label,center_ADC,center_ADC_err,center_keV,center_keV_err,sigma_ADC,sigma_ADC_err,sigma_keV,sigma_keV_err,amplitude_counts_per_s,amplitude_counts_per_s_err,gaussian_area_counts,gaussian_area_counts_err,fit_xmin,fit_xmax,background_slope,background_slope_err,background_intercept,background_intercept_err,fit_chi2,fit_redchi2,calibration_slope_keV_per_ADC,calibration_slope_err,calibration_offset_keV,calibration_offset_err,calibration_extra_sigma_keV,compton_region_length_ADC,compton_region_length_ADC_err,compton_region_length_keV,compton_region_length_keV_err,compton_region_slope_counts_per_s_per_ADC,compton_region_slope_err_per_ADC,compton_region_slope_counts_per_s_per_keV,compton_region_slope_err_per_keV,compton_region_intercept,compton_region_intercept_err,compton_region_redchi2,compton_region_average_drop_per_ADC,compton_region_average_drop_per_keV,compton_region_average_drop_per_keV_err
33,data/thickness_effects_behind/Pb thickness/Pb_...,mar3,0,compton_edge,350.787586,402.638137,458.271185,530.511167,42.581783,419.925515,56.104567,553.282154,33.107246,891.728691,3533.758199,101359.217385,287.43,350.79,-0.423569,8.109202,125.988367,1951.138798,1.796658,0.030977,1.317572,0.004624,-3.916731,2.043277,0.0,211.028241,402.664608,278.044911,530.550373,-0.028006,0.001347,-0.021256,0.001025,23.210952,0.347889,0.085153,-0.028006,-0.021256,0.001025
34,data/thickness_effects_in_front/Pb_11cmpmt_tos...,mar3,0,backscatter_peak,68.388465,3.147676,86.189999,4.634113,4.648058,3.467458,6.124152,4.568677,0.535360,0.363251,6.237451,6.289949,40.59,85.47,0.000408,0.006764,0.213711,0.389932,0.398241,0.009956,1.317572,0.004624,-3.916731,2.043277,0.0,76.481535,566.739731,100.769932,746.726385,0.001580,0.000639,0.001200,0.000485,0.167472,0.068039,0.037380,0.001580,0.001200,0.000485
35,data/thickness_effects_in_front/Pb_11cmpmt_tos...,mar3,0,compton_edge,144.870000,566.730990,186.959931,746.712005,17.672661,594.138508,23.285005,782.820292,3.043925,277.397234,134.842219,13097.877529,144.87,169.95,0.060419,6.051356,-11.145727,1154.463376,0.045457,0.002273,1.317572,0.004624,-3.916731,2.043277,0.0,76.481535,566.739731,100.769932,746.726385,0.001580,0.000639,0.001200,0.000485,0.167472,0.068039,0.037380,0.001580,0.001200,0.000485
36,data/thickness_effects_in_front/Pb_11cmpmt_tos...,mar3,0,backscatter_peak,67.379317,3.861092,84.860374,5.491115,4.779118,4.431827,6.296833,5.839294,0.617437,0.479455,7.396564,8.946280,30.03,85.47,0.005411,0.008662,0.303660,0.432197,0.271019,0.005420,1.317572,0.004624,-3.916731,2.043277,0.0,87.106904,5.510347,114.769621,7.852960,0.008833,0.001158,0.006704,0.000879,0.038104,0.125852,0.062872,0.008833,0.006704,0.000879
37,data/thickness_effects_in_front/Pb_11cmpmt_tos...,mar3,0,compton_edge,154.486221,3.931398,199.629995,5.613968,10.316312,5.196814,13.592484,6.847343,1.413488,0.487555,36.551621,22.315595,127.71,183.15,-0.006157,0.011142,1.690612,1.914348,0.748876,0.014684,1.317572,0.004624,-3.916731,2.043277,0.0,87.106904,5.510347,114.769621,7.852960,0.008833,0.001158,0.006704,0.000879,0.038104,0.125852,0.062872,0.008833,0.006704,0.000879


In [273]:

# -----------------------------
# CLEAR INTERACTIVE STATE
# -----------------------------

clear_feature_state()
print("Interactive state cleared.")

Interactive state cleared.


In [234]:

# -----------------------------
# FULL CLEANUP / RESET
# -----------------------------

def clear_all_compton_analysis_outputs(
    remove_output_directory=True,
    clear_interactive_state=True,
    clear_runtime_globals=False,
):
    """
    Use this when you want to restart Compton analysis cleanly.
    """
    if remove_output_directory and OUTPUT_ROOT.exists():
        shutil.rmtree(OUTPUT_ROOT)
        print(f"Removed: {OUTPUT_ROOT}")

    if clear_interactive_state:
        clear_feature_state()
        print("Cleared interactive selector state.")

    if clear_runtime_globals:
        for name in [
            "current_file",
            "current_rel",
            "x_current",
            "raw_rate_current",
            "background_rate_current",
            "adjusted_rate_current",
            "sample_meta_current",
            "background_meta_current",
            "feature_specs_current",
            "fit_results_by_label",
            "results_df",
        ]:
            if name in globals():
                del globals()[name]
        print("Cleared runtime globals.")

# Uncomment only when you really want a hard reset.
# clear_all_compton_analysis_outputs(
#     remove_output_directory=True,
#     clear_interactive_state=True,
#     clear_runtime_globals=True,
# )